In [1]:
from transformers import Pix2StructProcessor, Pix2StructForConditionalGeneration
import requests
from PIL import Image
import os
import json
import pandas as pd
import torch
from tqdm import tqdm

In [2]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

processor = Pix2StructProcessor.from_pretrained('google/deplot')
model = Pix2StructForConditionalGeneration.from_pretrained('google/deplot').to(device)

In [ ]:
types = ['t1', 't2', 't3']
directories = ['../../type1/simple', '../../type2/simple', '../../type3/simple']

os.makedirs('../../model_responses/chart_to_table/deplot', exist_ok=True)

for i in range(len(types)):
    chart_type = types[i]
    directory = directories[i]
    
    all_charts = json.load(open(f'{chart_type}_charts.json'))    
    
    os.makedirs(f'../../model_responses/chart_to_table/deplot/{chart_type}', exist_ok=True)
    
    for key, value in tqdm(all_charts.items()):
        chart_name = key
        charts = value
        
        images = [Image.open(f'{directory}/{chart}') for chart in charts]
        texts = ["Generate underlying data table of the figure below:"] * len(images)
        
        inputs = processor(images, texts, return_tensors="pt", padding=True).to(device)
        preds = model.generate(**inputs, max_new_tokens = 1024)
        tables = processor.batch_decode(preds, skip_special_tokens=True)
    
        with open(f'../../model_responses/chart_to_table/deplot/{chart_type}/{chart_name}.json', 'w') as f:
            json.dump(tables, f, indent=4)

        del images, texts, inputs, preds, tables
    
    

In [3]:
img_path_1 = "../../type1/simple/multichart_5_0.png"
img_path_2 = "../../type1/simple/multichart_5_1.png"
img_path_3 = "../../type1/simple/multichart_5_2.png"
# image = Image.open(img_path_1)

# inputs = processor(images=image, text="Generate underlying data table of the figure below:", return_tensors="pt").to(device)
# predictions = model.generate(**inputs, max_new_tokens=512)
# print('\n'.join(processor.decode(predictions[0], skip_special_tokens=True).split('<0x0A>')))

In [4]:
images = [Image.open(img_path_1), Image.open(img_path_2), Image.open(img_path_3)]
texts = ["Generate underlying data table of the figure below:"]*3
inputs = processor(images=images, text=texts, return_tensors="pt", padding=True).to(device)
predictions = model.generate(**inputs, max_new_tokens=512)
# for i, pred in enumerate(predictions):
#     table_text = '\n'.join(processor.decode(pred, skip_special_tokens=True).split('<0x0A>'))
#     print(f"Example {i}: \n {table_text}")
#     print()

In [5]:
pred = processor.batch_decode(predictions, skip_special_tokens=True)

In [ ]:
pred